# T12 vertebral-body midpoint audit (Colab)

**Notebook v1.0.2** | 2026-09-11 | Fix TotalSegmentator 2.18 task compatibility and fail-fast probe

Runs the repository's validated `run_pilot.py` unchanged in scientific logic, with Drive-backed checkpointing and per-case local scratch.

**Runtime:** T4 GPU + High-RAM is the cheapest fitting tier. The local run measured a host-RAM peak of approximately 15.7 GiB, so standard 12.7 GiB RAM is unsafe; require at least 20 GiB host RAM. VRAM use was below 1 GiB. Runtime and CU estimates are reported from the smoke test rather than guessed here.

Keep this notebook open in Chrome. Do not run long jobs through the VS Code Colab plugin. Important state is written to Drive; `/content` is disposable scratch.

## 📜 Changelog

| Version | Date | Change | Commit |
|---|---|---|---|
| `v1.0.0` | 2026-09-11 | Initial launcher; Drive inventory reuse, crash-safe checkpoint, second-session resume gate | `a1beb9a` |
| `v1.0.1` | 2026-09-11 | Activate licensed TotalSegmentator tasks from a case-tolerant Colab Secret | `6c7abe6` |
| `v1.0.2` ⬅️ **current** | 2026-09-11 | Pin validated TotalSegmentator 2.18.0 and fail before compute if either task is unavailable | `uncommitted` |

> `NOTEBOOK_VERSION` is asserted in Cell 1. A mismatch means a cached notebook copy; reopen with `?flush_cache=true`.


In [ ]:
# Cell 1: Environment setup, Drive mount, and complete input validation
import csv, importlib.metadata, json, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

pip_cmd = [sys.executable, "-m", "pip", "install", "-q", "TotalSegmentator==2.18.0", "SimpleITK", "scipy", "psutil"]
pip_result = subprocess.run(pip_cmd)
assert pip_result.returncode == 0, "BLOCKING: dependency installation failed"

from google.colab import userdata
license_number = None
for secret_name in ("totalseg_license", "TOTALSEG_LICENSE", "TOTALSEGMENTATOR_LICENSE"):
    try:
        license_number = userdata.get(secret_name)
    except Exception:
        license_number = None
    if license_number:
        break
assert license_number, "BLOCKING: add TOTALSEG_LICENSE (or totalseg_license) in Colab Secrets and enable notebook access"
license_tool = shutil.which("totalseg_set_license")
assert license_tool, "BLOCKING: totalseg_set_license CLI missing"
license_result = subprocess.run([license_tool, "-l", license_number], text=True, capture_output=True)
assert license_result.returncode == 0, "BLOCKING: TotalSegmentator license activation failed; verify the Secret value"
del license_number
print("TotalSegmentator licensed-task access configured from Colab Secret.", flush=True)

NOTEBOOK_VERSION = "1.0.2"
HEADER_VERSION = "1.0.2"
assert NOTEBOOK_VERSION == HEADER_VERSION
print(f"Notebook version: {NOTEBOOK_VERSION}", flush=True)

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

GDRIVE = Path("/content/drive/MyDrive/cardiac_colab")
TASK_DIR = GDRIVE / "t12_midpoint_audit_20260911"
ASSET_DIR = TASK_DIR / "assets"
OUTPUT_DIR = TASK_DIR / "output"
LOG_DIR = TASK_DIR / "logs"
CHECKPOINT_FILE = OUTPUT_DIR / "results.csv"
MANIFEST = ASSET_DIR / "full_manifest_colab_20260911.csv"
RUNNER = ASSET_DIR / "run_pilot.py"
LABEL_ARCHIVE = ASSET_DIR / "t12_existing_labels_20260911.tar.gz"
SCRATCH_ROOT = Path("/mnt/local-scratch") if Path("/mnt/local-scratch").is_dir() else Path("/content")
WORK_DIR = SCRATCH_ROOT / "t12_midpoint_audit"
LABEL_DIR = Path("/content/t12_audit_labels")
for path in (OUTPUT_DIR, LOG_DIR, WORK_DIR): path.mkdir(parents=True, exist_ok=True)

required = [MANIFEST, RUNNER, LABEL_ARCHIVE]
missing_assets = [str(p) for p in required if not p.is_file()]
assert not missing_assets, "BLOCKING missing assets:\n" + "\n".join(missing_assets)

if not LABEL_DIR.is_dir():
    print(f"Extracting labels to {LABEL_DIR} ...", flush=True)
    with tarfile.open(LABEL_ARCHIVE, "r:gz") as tf: tf.extractall("/content")

rows = list(csv.DictReader(MANIFEST.open()))
missing_inputs = []
for row in rows:
    for p in (Path(row["image_path"]), Path(row["label_dir"])/"vertebrae_T12.nii.gz", Path(row["label_dir"])/"tissue_4types_torso_fat.nii.gz"):
        if not p.is_file(): missing_inputs.append(str(p))
print(f"Manifest rows: {len(rows)}; missing required files: {len(missing_inputs)}", flush=True)
if missing_inputs:
    print("\n".join(missing_inputs[:50]), flush=True)
assert not missing_inputs, f"BLOCKING: {len(missing_inputs)} required inputs are missing"


In [ ]:
# Cell 2: GPU, host RAM, disk, and tier gate
import psutil, torch
assert torch.cuda.is_available(), "BLOCKING: select a GPU runtime"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
ram_gb = psutil.virtual_memory().total / 2**30
disk = shutil.disk_usage(SCRATCH_ROOT)
print(f"GPU={gpu_name}; VRAM={vram_gb:.1f} GiB; RAM={ram_gb:.1f} GiB; scratch_free={disk.free/2**30:.1f} GiB", flush=True)
assert ram_gb >= 20, "BLOCKING: measured local peak requires Colab High-RAM (>=20 GiB)"
assert disk.free / 2**30 >= 20, "BLOCKING: insufficient local scratch"
if "A100" in gpu_name: print("[WARN] T4 High-RAM is sufficient and cheaper for this GPU-light inference.", flush=True)


In [ ]:
# Cell 3: install dependencies and define live logging helpers
assert shutil.which("TotalSegmentator"), "BLOCKING: TotalSegmentator CLI missing"
help_result = subprocess.run(["TotalSegmentator", "--help"], text=True, capture_output=True)
assert help_result.returncode == 0 and "--device" in help_result.stdout, "BLOCKING: expected CLI flags missing"
task_result = subprocess.run(["TotalSegmentator", "--list-tasks"], text=True, capture_output=True)
assert task_result.returncode == 0, "BLOCKING: TotalSegmentator task listing failed"
for required_task in ("vertebrae_body", "vertebrae_pp"):
    assert required_task in task_result.stdout, f"BLOCKING: {required_task} unavailable in installed TotalSegmentator"
print("TotalSegmentator", importlib.metadata.version("TotalSegmentator"), flush=True)

def run_streamed(cmd, log_path, drive_log, env=None):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    line_count = 0
    with log_path.open("a") as logf:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
        assert proc.stdout is not None
        try:
            for line in proc.stdout:
                print(line, end="")
                logf.write(line); logf.flush(); line_count += 1
                if line_count % 20 == 0:
                    shutil.copy2(log_path, drive_log)
        finally:
            shutil.copy2(log_path, drive_log)
        rc = proc.wait()
    assert rc == 0, f"subprocess failed with return code {rc}"
    return rc


In [ ]:
# Cell 4: restore TotalSegmentator task-weight cache and verify model entry point
TS_CACHE_ARCHIVE = GDRIVE / "cache" / "ts_weights_vertebrae_body_pp.tar.gz"
TS_HOME = Path.home() / ".totalsegmentator"
if TS_CACHE_ARCHIVE.is_file() and not TS_HOME.exists():
    print("Restoring cached TotalSegmentator weights...", flush=True)
    with tarfile.open(TS_CACHE_ARCHIVE, "r:gz") as tf: tf.extractall(Path.home())
else:
    print("Weight cache absent or already restored; smoke test will verify/download exact task weights.", flush=True)
print("Model CLI:", shutil.which("TotalSegmentator"), flush=True)


In [ ]:
# Cell 5: BLOCKING real one-case smoke test
SMOKE_DIR = WORK_DIR / "smoke"
if SMOKE_DIR.exists(): shutil.rmtree(SMOKE_DIR)
SMOKE_DIR.mkdir(parents=True)
smoke_manifest = SMOKE_DIR / "smoke_manifest.csv"
with MANIFEST.open() as src, smoke_manifest.open("w", newline="") as dst:
    reader = csv.DictReader(src); writer = csv.DictWriter(dst, fieldnames=reader.fieldnames); writer.writeheader(); writer.writerow(next(reader))
smoke_checkpoint = SMOKE_DIR / "results.csv"
smoke_log = SMOKE_DIR / "smoke.log"
smoke_drive_log = LOG_DIR / "smoke_latest.log"
t0 = time.time()
run_streamed([sys.executable, "-u", str(RUNNER), "--manifest", str(smoke_manifest), "--out", str(SMOKE_DIR/"work"), "--checkpoint", str(smoke_checkpoint), "--limit", "1"], smoke_log, smoke_drive_log)
smoke_rows = list(csv.DictReader(smoke_checkpoint.open()))
assert len(smoke_rows) == 1 and smoke_rows[0]["status"] == "SUCCESS", f"BLOCKING smoke failed: {smoke_rows}"
smoke_minutes = (time.time()-t0)/60
print(f"BLOCKING smoke passed in {smoke_minutes:.1f} min/case; projected 718-case upper-bound={smoke_minutes*718/60:.1f} h", flush=True)
if not TS_CACHE_ARCHIVE.exists() and TS_HOME.exists():
    TS_CACHE_ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
    with tarfile.open(TS_CACHE_ARCHIVE, "w:gz") as tf: tf.add(TS_HOME, arcname=".totalsegmentator")
    print(f"Saved task-weight cache: {TS_CACHE_ARCHIVE}", flush=True)


In [ ]:
# Cell 6: checkpointed full batch with live output and Drive log mirroring
RUN_LOG = WORK_DIR / "batch.log"
DRIVE_RUN_LOG = LOG_DIR / "batch_latest.log"
cmd = [sys.executable, "-u", str(RUNNER), "--manifest", str(MANIFEST), "--out", str(WORK_DIR/"cases"), "--checkpoint", str(CHECKPOINT_FILE)]
started = time.time(); batch_loop_completed = False
try:
    run_streamed(cmd, RUN_LOG, DRIVE_RUN_LOG)
    batch_loop_completed = True
finally:
    if RUN_LOG.exists(): shutil.copy2(RUN_LOG, DRIVE_RUN_LOG)
elapsed_min = (time.time()-started)/60
print(f"Batch cell ended; elapsed={elapsed_min:.1f} min; checkpoint={CHECKPOINT_FILE}", flush=True)


In [ ]:
# Cell 7: validation, second-session resume rehearsal, result link, auto-disconnect
result_rows = list(csv.DictReader(CHECKPOINT_FILE.open()))
success = [r for r in result_rows if r.get("status") == "SUCCESS"]
failed = [r for r in result_rows if r.get("status") != "SUCCESS"]
print(f"Results: total={len(result_rows)} success={len(success)} failed={len(failed)}", flush=True)
assert len({r["patientingroupid"] for r in success}) == len(success), "duplicate successful checkpoint rows"

# Fresh-directory second-session rehearsal: completed rows must be skipped and no CT staged.
REHEARSAL = WORK_DIR / "resume_rehearsal"
if REHEARSAL.exists(): shutil.rmtree(REHEARSAL)
REHEARSAL.mkdir(parents=True)
resume_probe = subprocess.run([sys.executable, "-u", str(RUNNER), "--manifest", str(MANIFEST), "--out", str(REHEARSAL), "--checkpoint", str(CHECKPOINT_FILE), "--limit", "1"], text=True, capture_output=True)
assert resume_probe.returncode == 0, resume_probe.stdout + resume_probe.stderr
assert not any(REHEARSAL.iterdir()), "BLOCKING: completed case was unexpectedly restaged"
print("Second-session resume rehearsal passed: completed case skipped; no historical outputs restored.", flush=True)
print(f"Results: {CHECKPOINT_FILE}", flush=True)
print(f"Estimated compute used: {elapsed_min/60*1.67:.2f} CU at T4 1.67 CU/h (batch cell only)", flush=True)

if batch_loop_completed:
    print("Auto-disconnecting in 30 seconds; results and logs are safe on Drive.", flush=True)
    time.sleep(30)
    from google.colab import runtime
    runtime.unassign()
